<a href="https://colab.research.google.com/github/AxdKyn13/Projects-Trial/blob/main/Code_FPCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===========================================
# 1. MOUNT DRIVE & SPLIT DATASET
# ===========================================
from google.colab import drive
drive.mount('/content/drive')

input_folder = "/content/drive/MyDrive/Colab Notebooks/dataset_kucing"
output_folder = "/content/drive/MyDrive/Colab Notebooks/dataset_kucing_split"

!rm -rf "/content/drive/MyDrive/Colab Notebooks/dataset_kucing_split"

import splitfolders

# SPLIT 80% TRAIN — 10% VAL — 10% TEST
splitfolders.ratio(
    input_folder,
    output=output_folder,
    seed=42,
    ratio=(0.8, 0.1, 0.1)
)


# ===========================================
# 2. IMPORT LIBRARY
# ===========================================
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pandas as pd
import os


# ===========================================
# 3. DATA GENERATOR (Train, Val, Test)
# ===========================================
train_dir = f"{output_folder}/train"
val_dir   = f"{output_folder}/val"
test_dir  = f"{output_folder}/test"

train_datagen = ImageDataGenerator(
    rescale=1/255.0,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1/255.0)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical"
)

val_gen = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)

classes = list(train_gen.class_indices.keys())
print("Classes:", classes)


# ===========================================
# 4. DEFINISI MODEL
# ===========================================
base = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

x = GlobalAveragePooling2D()(base.output)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(len(classes), activation="softmax")(x)

model = Model(inputs=base.input, outputs=output)

model.compile(
    optimizer=Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


# ===========================================
# 5. TRAINING (latihan model)
# ===========================================
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=[early_stop]
)


# ===========================================
# 6. TESTING (model dipakai untuk memprediksi test set)
# ===========================================
print("\nTESTING")
test_gen.reset()

y_pred_prob_test = model.predict(test_gen)
y_pred_test = np.argmax(y_pred_prob_test, axis=1)
y_true_test = test_gen.classes

# ===========================================
# 7. EVALUASI TRAINING (VALIDATION SET)
# ===========================================
print("\nEvaluasi Validation")

# Prediksi val
val_gen.reset()
y_pred_val = np.argmax(model.predict(val_gen), axis=1)
y_true_val = val_gen.classes

cm_val = confusion_matrix(y_true_val, y_pred_val)

plt.figure(figsize=(6,5))
sns.heatmap(cm_val, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix Validation")
plt.xlabel("Prediksi")
plt.ylabel("Label Asli")
plt.show()

print("\nClassification Report Validation :")
print(classification_report(y_true_val, y_pred_val, target_names=classes))

# ===========================================
# 8. EVALUASI TESTING (TEST SET)
# ===========================================
print("\nEVALUASI TESTING")

cm_test = confusion_matrix(y_true_test, y_pred_test)

plt.figure(figsize=(6,5))
sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix Testing")
plt.xlabel("Prediksi")
plt.ylabel("Label Asli")
plt.show()

print("\nClassification Report Testing :")
print(classification_report(y_true_test, y_pred_test, target_names=classes))

test_loss, test_acc = model.evaluate(test_gen)
print(f"\nAkurasi Testing : {test_acc:.4f}")